In [1]:
"""
===================================

Definition:
  PASS = clean barcode crop extracted from source image after quality filtering
  FAIL = same barcode crop with synthetic physical damage applied

Important:
  - pyzbar is OPTIONAL
  - if pyzbar is available, it is used ONLY for barcode localization
  - pyzbar is NOT used for PASS/FAIL labelling
  - if pyzbar is unavailable, OpenCV morphology fallback is used

Dataset output:
  barcode_dataset_hf_final/
    train/
      PASS/
      FAIL/
    validation/
      PASS/
      FAIL/
    test/
      PASS/
      FAIL/
    manifest.json

Install:
  pip install datasets pillow opencv-python numpy tqdm

Optional:
  pip install pyzbar

Run:
  python prepare_barcode_dataset_hf_final.py
"""

import os
import json
import random
import shutil
from typing import Optional, Tuple, List

import cv2
import numpy as np
from PIL import Image
from tqdm import tqdm


# ============================================================
# Config
# ============================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

HF_DATASET_NAME = "amaye15/barcodes-google-ocr"
HF_SPLIT = "train"

TARGET_PASS_IMAGES = 3000
MAX_STREAM_IMAGES = 20000

FINAL_IMG_SIZE = (224, 224)
OUTPUT_DIR = "barcode_dataset_hf_final"
TMP_PASS_DIR = "_tmp_pass_crops"
TMP_FAIL_DIR = "_tmp_fail_crops"

SPLIT_RATIOS = (0.70, 0.15, 0.15)

MIN_BARCODE_AREA_RATIO = 0.08
MAX_BARCODE_AREA_RATIO = 0.95
MIN_ASPECT_RATIO = 1.2
MAX_ASPECT_RATIO = 18.0
MIN_CROP_WIDTH = 40
MIN_CROP_HEIGHT = 20
MIN_SHARPNESS = 35.0
MIN_STRIPE_SCORE = 1.15
MIN_MEAN_INTENSITY = 35
MAX_MEAN_INTENSITY = 245

PAD_X_RATIO = 0.08
PAD_Y_RATIO = 0.15

JPEG_QUALITY = 95


# ============================================================
# Optional pyzbar import
# ============================================================

PYZBAR_AVAILABLE = False
try:
    from pyzbar.pyzbar import decode
    PYZBAR_AVAILABLE = True
except Exception:
    PYZBAR_AVAILABLE = False


# ============================================================
# Directory helpers
# ============================================================

def reset_dirs():
    for d in [TMP_PASS_DIR, TMP_FAIL_DIR, OUTPUT_DIR]:
        if os.path.exists(d):
            shutil.rmtree(d)
        os.makedirs(d, exist_ok=True)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


# ============================================================
# Basic image helpers
# ============================================================

def pil_to_bgr(pil_img: Image.Image) -> np.ndarray:
    return cv2.cvtColor(np.array(pil_img.convert("RGB")), cv2.COLOR_RGB2BGR)


def bgr_to_pil(img_bgr: np.ndarray) -> Image.Image:
    return Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))


def resize_final(pil_img: Image.Image) -> Image.Image:
    return pil_img.convert("RGB").resize(FINAL_IMG_SIZE, Image.LANCZOS)


def clamp(val: int, lo: int, hi: int) -> int:
    return max(lo, min(val, hi))


def clamp_box(x1: int, y1: int, x2: int, y2: int, w: int, h: int) -> Tuple[int, int, int, int]:
    x1 = clamp(x1, 0, w - 1)
    y1 = clamp(y1, 0, h - 1)
    x2 = clamp(x2, x1 + 1, w)
    y2 = clamp(y2, y1 + 1, h)
    return x1, y1, x2, y2


def expand_box(
    x1: int, y1: int, x2: int, y2: int, w: int, h: int,
    pad_x_ratio: float = PAD_X_RATIO,
    pad_y_ratio: float = PAD_Y_RATIO
) -> Tuple[int, int, int, int]:
    bw = x2 - x1
    bh = y2 - y1
    pad_x = int(round(bw * pad_x_ratio))
    pad_y = int(round(bh * pad_y_ratio))
    return clamp_box(x1 - pad_x, y1 - pad_y, x2 + pad_x, y2 + pad_y, w, h)


# ============================================================
# Barcode localization
# ============================================================

def detect_barcode_pyzbar(img_bgr: np.ndarray) -> Optional[Tuple[int, int, int, int]]:
    """
    Optional pyzbar-based barcode localization.
    Used ONLY to find barcode bounding box.
    """
    if not PYZBAR_AVAILABLE:
        return None

    try:
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        results = decode(img_rgb)

        if not results:
            return None

        best_box = None
        best_area = -1

        for r in results:
            rect = r.rect
            x, y, w, h = rect.left, rect.top, rect.width, rect.height
            area = w * h
            if area > best_area:
                best_area = area
                best_box = (x, y, x + w, y + h)

        return best_box
    except Exception:
        return None


def detect_barcode_morphology(img_bgr: np.ndarray) -> Optional[Tuple[int, int, int, int]]:
    """
    OpenCV fallback barcode localization.
    Looks for dense vertical stripe regions.
    """
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape[:2]

    blur = cv2.GaussianBlur(gray, (3, 3), 0)

    grad_x = cv2.Sobel(blur, cv2.CV_32F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(blur, cv2.CV_32F, 0, 1, ksize=3)
    grad = cv2.convertScaleAbs(grad_x - 0.5 * grad_y)

    _, thresh = cv2.threshold(grad, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    kernel_close = cv2.getStructuringElement(cv2.MORPH_RECT, (21, 7))
    closed = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel_close)

    kernel_open = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    cleaned = cv2.morphologyEx(closed, cv2.MORPH_OPEN, kernel_open)

    contours, _ = cv2.findContours(cleaned, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None

    image_area = w * h
    candidates = []

    for cnt in contours:
        x, y, bw, bh = cv2.boundingRect(cnt)
        area = bw * bh
        if area < 0.01 * image_area:
            continue

        aspect = bw / max(bh, 1)
        if aspect < 1.2:
            continue

        roi = cleaned[y:y + bh, x:x + bw]
        fill_ratio = float(np.count_nonzero(roi)) / max(1, roi.size)

        if fill_ratio < 0.15 or fill_ratio > 0.95:
            continue

        score = area * min(aspect, 8.0)
        candidates.append((score, (x, y, x + bw, y + bh)))

    if not candidates:
        return None

    candidates.sort(key=lambda z: z[0], reverse=True)
    return candidates[0][1]


def detect_barcode_box(img_bgr: np.ndarray) -> Optional[Tuple[int, int, int, int]]:
    box = detect_barcode_pyzbar(img_bgr)
    if box is not None:
        return box
    return detect_barcode_morphology(img_bgr)


# ============================================================
# Quality filtering
# ============================================================

def laplacian_sharpness(gray: np.ndarray) -> float:
    return float(cv2.Laplacian(gray, cv2.CV_64F).var())


def barcode_stripe_score(gray: np.ndarray) -> float:
    """
    Higher value means stronger vertical stripe energy.
    Good for 1D barcodes.
    """
    sobel_x = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    ex = float(np.mean(np.abs(sobel_x)))
    ey = float(np.mean(np.abs(sobel_y))) + 1e-6
    return ex / ey


def crop_quality_ok(
    full_img_bgr: np.ndarray,
    crop_bgr: np.ndarray,
    box: Tuple[int, int, int, int]
) -> bool:
    fh, fw = full_img_bgr.shape[:2]
    x1, y1, x2, y2 = box
    bw = x2 - x1
    bh = y2 - y1

    if bw < MIN_CROP_WIDTH or bh < MIN_CROP_HEIGHT:
        return False

    area_ratio = (bw * bh) / float(max(1, fw * fh))
    if area_ratio < MIN_BARCODE_AREA_RATIO or area_ratio > MAX_BARCODE_AREA_RATIO:
        return False

    aspect = bw / max(bh, 1)
    if aspect < MIN_ASPECT_RATIO or aspect > MAX_ASPECT_RATIO:
        return False

    gray = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2GRAY)

    sharp = laplacian_sharpness(gray)
    if sharp < MIN_SHARPNESS:
        return False

    stripe = barcode_stripe_score(gray)
    if stripe < MIN_STRIPE_SCORE:
        return False

    mean_intensity = float(np.mean(gray))
    if mean_intensity < MIN_MEAN_INTENSITY or mean_intensity > MAX_MEAN_INTENSITY:
        return False

    return True


def extract_barcode_crop(pil_img: Image.Image) -> Optional[Image.Image]:
    img_bgr = pil_to_bgr(pil_img)
    h, w = img_bgr.shape[:2]

    box = detect_barcode_box(img_bgr)
    if box is None:
        return None

    x1, y1, x2, y2 = expand_box(*box, w=w, h=h)
    crop = img_bgr[y1:y2, x1:x2]

    if crop.size == 0:
        return None

    if not crop_quality_ok(img_bgr, crop, (x1, y1, x2, y2)):
        return None

    return resize_final(bgr_to_pil(crop))


# ============================================================
# Synthetic physical damage functions
# ============================================================

def add_scratches(a: np.ndarray) -> np.ndarray:
    h, w = a.shape[:2]
    for _ in range(random.randint(3, 8)):
        x1 = random.randint(0, w - 1)
        x2 = random.randint(0, w - 1)
        y = random.randint(h // 5, 4 * h // 5)
        thickness = random.randint(1, 3)
        color = random.choice([(0, 0, 0), (70, 70, 70), (210, 210, 210)])
        cv2.line(a, (x1, y), (x2, y), color, thickness)
    return a


def add_diagonal_scratches(a: np.ndarray) -> np.ndarray:
    h, w = a.shape[:2]
    for _ in range(random.randint(2, 5)):
        x1 = random.randint(0, w - 1)
        y1 = random.randint(0, h - 1)
        x2 = clamp(x1 + random.randint(-w // 3, w // 3), 0, w - 1)
        y2 = clamp(y1 + random.randint(-h // 3, h // 3), 0, h - 1)
        thickness = random.randint(1, 3)
        color = random.choice([(0, 0, 0), (60, 60, 60)])
        cv2.line(a, (x1, y1), (x2, y2), color, thickness)
    return a


def add_ink_smudge(a: np.ndarray) -> np.ndarray:
    h, w = a.shape[:2]
    overlay = a.copy()

    cx = random.randint(w // 4, 3 * w // 4)
    cy = random.randint(h // 4, 3 * h // 4)
    rw = random.randint(w // 10, w // 4)
    rh = random.randint(h // 8, h // 3)

    cv2.ellipse(
        overlay,
        (cx, cy),
        (rw, rh),
        random.randint(0, 180),
        0,
        360,
        (random.randint(0, 70), random.randint(0, 70), random.randint(0, 70)),
        -1
    )

    alpha = random.uniform(0.35, 0.75)
    return cv2.addWeighted(overlay, alpha, a, 1 - alpha, 0)


def add_partial_cover(a: np.ndarray) -> np.ndarray:
    h, w = a.shape[:2]
    x_start = random.randint(w // 5, 3 * w // 5)
    cover_w = random.randint(w // 10, w // 4)
    color = random.choice([
        (255, 255, 255),
        (245, 245, 210),
        (215, 215, 215)
    ])
    a[:, x_start:x_start + cover_w] = color
    return a


def add_print_degradation(a: np.ndarray) -> np.ndarray:
    h, w = a.shape[:2]
    for _ in range(random.randint(5, 12)):
        x = random.randint(0, w - 2)
        fade_w = random.randint(1, 3)
        fade_amount = random.randint(70, 150)
        strip = a[:, x:x + fade_w].astype(np.int32)
        strip = np.clip(strip + fade_amount, 0, 255).astype(np.uint8)
        a[:, x:x + fade_w] = strip
    return a


def add_water_damage(a: np.ndarray) -> np.ndarray:
    h, w = a.shape[:2]
    map_x = np.zeros((h, w), dtype=np.float32)
    map_y = np.zeros((h, w), dtype=np.float32)

    amplitude = random.randint(2, 6)
    frequency = random.uniform(0.04, 0.12)

    for i in range(h):
        for j in range(w):
            map_x[i, j] = j + amplitude * np.sin(2 * np.pi * i * frequency)
            map_y[i, j] = i + amplitude * np.cos(2 * np.pi * j * frequency)

    warped = cv2.remap(a, map_x, map_y, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)

    dark_mask = np.random.random((h, w)) > 0.88
    warped[dark_mask] = np.clip(
        warped[dark_mask].astype(np.int32) - random.randint(25, 60),
        0,
        255
    ).astype(np.uint8)

    return warped


def add_torn_edge(a: np.ndarray) -> np.ndarray:
    h, w = a.shape[:2]
    bg_color = (255, 255, 255)
    style = random.choice(["corner", "edge", "bottom"])

    if style == "corner":
        corner = random.choice(["tl", "tr", "bl", "br"])
        depth_x = random.randint(w // 6, w // 3)
        depth_y = random.randint(h // 6, h // 3)
        n_points = random.randint(8, 14)

        if corner == "tr":
            xs = np.linspace(w - depth_x, w, n_points).astype(int)
            ys = np.linspace(0, depth_y, n_points).astype(int)
            xs = np.clip(xs + np.random.randint(-5, 5, n_points), 0, w)
            ys = np.clip(ys + np.random.randint(-5, 5, n_points), 0, h)
            pts = np.array(list(zip(xs, ys)) + [(w, 0)], dtype=np.int32)

        elif corner == "tl":
            xs = np.linspace(0, depth_x, n_points).astype(int)
            ys = np.linspace(depth_y, 0, n_points).astype(int)
            xs = np.clip(xs + np.random.randint(-5, 5, n_points), 0, w)
            ys = np.clip(ys + np.random.randint(-5, 5, n_points), 0, h)
            pts = np.array([(0, 0)] + list(zip(xs, ys)), dtype=np.int32)

        elif corner == "br":
            xs = np.linspace(w, w - depth_x, n_points).astype(int)
            ys = np.linspace(h - depth_y, h, n_points).astype(int)
            xs = np.clip(xs + np.random.randint(-5, 5, n_points), 0, w)
            ys = np.clip(ys + np.random.randint(-5, 5, n_points), 0, h)
            pts = np.array(list(zip(xs, ys)) + [(w, h)], dtype=np.int32)

        else:
            xs = np.linspace(depth_x, 0, n_points).astype(int)
            ys = np.linspace(h, h - depth_y, n_points).astype(int)
            xs = np.clip(xs + np.random.randint(-5, 5, n_points), 0, w)
            ys = np.clip(ys + np.random.randint(-5, 5, n_points), 0, h)
            pts = np.array([(0, h)] + list(zip(xs, ys)), dtype=np.int32)

        cv2.fillPoly(a, [pts.reshape(-1, 1, 2)], bg_color)

    elif style == "edge":
        side = random.choice(["left", "right"])
        depth = random.randint(w // 8, w // 4)
        n_pts = random.randint(10, 18)
        ys = np.linspace(0, h, n_pts).astype(int)

        if side == "right":
            xs = np.clip(w - depth + np.random.randint(-8, 8, n_pts), 0, w)
            pts = np.array([(w, 0)] + list(zip(xs, ys)) + [(w, h)], dtype=np.int32)
        else:
            xs = np.clip(depth + np.random.randint(-8, 8, n_pts), 0, w)
            pts = np.array([(0, 0)] + list(zip(xs, ys)) + [(0, h)], dtype=np.int32)

        cv2.fillPoly(a, [pts.reshape(-1, 1, 2)], bg_color)

    else:
        depth = random.randint(h // 6, h // 3)
        n_pts = random.randint(10, 18)
        xs = np.linspace(0, w, n_pts).astype(int)
        ys = np.clip(h - depth + np.random.randint(-8, 8, n_pts), 0, h)
        pts = np.array([(0, h)] + list(zip(xs, ys)) + [(w, h)], dtype=np.int32)
        cv2.fillPoly(a, [pts.reshape(-1, 1, 2)], bg_color)

    return a


def add_missing_middle(a: np.ndarray) -> np.ndarray:
    h, w = a.shape[:2]
    style = random.choice(["h_band", "v_gap", "shatter"])

    if style == "h_band":
        band_h = random.randint(h // 6, h // 3)
        band_y = random.randint(h // 4, h // 2)
        fill_style = random.choice(["white", "grey", "noise"])

        if fill_style == "white":
            a[band_y:band_y + band_h, :] = 255
        elif fill_style == "grey":
            val = random.randint(170, 220)
            a[band_y:band_y + band_h, :] = (val, val, val)
        else:
            noise = np.random.randint(190, 255, (band_h, w, 3), dtype=np.uint8)
            a[band_y:band_y + band_h, :] = noise

    elif style == "v_gap":
        for _ in range(random.randint(1, 3)):
            gap_x = random.randint(w // 6, 4 * w // 5)
            gap_w = random.randint(w // 12, w // 5)
            a[:, gap_x:gap_x + gap_w] = 255

    else:
        for _ in range(random.randint(2, 4)):
            cx = random.randint(w // 4, 3 * w // 4)
            cy = random.randint(h // 4, 3 * h // 4)
            rw = random.randint(w // 10, w // 4)
            rh = random.randint(h // 8, h // 3)

            n_pts = random.randint(6, 10)
            angles = np.linspace(0, 2 * np.pi, n_pts, endpoint=False)
            r_var = np.random.uniform(0.6, 1.0, n_pts)

            pts_x = (cx + rw * r_var * np.cos(angles)).astype(int)
            pts_y = (cy + rh * r_var * np.sin(angles)).astype(int)

            pts_x = np.clip(pts_x, 0, w - 1)
            pts_y = np.clip(pts_y, 0, h - 1)

            pts = np.stack([pts_x, pts_y], axis=1).reshape(-1, 1, 2)
            fill = random.choice([255, 235, 210])
            cv2.fillPoly(a, [pts], (fill, fill, fill))

    return a


DAMAGE_FNS = [
    ("scratches", add_scratches),
    ("diagonal_scratches", add_diagonal_scratches),
    ("ink_smudge", add_ink_smudge),
    ("partial_cover", add_partial_cover),
    ("print_degradation", add_print_degradation),
    ("water_damage", add_water_damage),
    ("torn_edge", add_torn_edge),
    ("missing_middle", add_missing_middle),
]


def apply_damage(pil_img: Image.Image) -> Tuple[Image.Image, List[str]]:
    a = np.array(pil_img.convert("RGB"))
    a = cv2.cvtColor(a, cv2.COLOR_RGB2BGR)

    chosen = random.sample(DAMAGE_FNS, k=random.randint(1, 2))
    damage_names = []

    for name, fn in chosen:
        a = fn(a)
        damage_names.append(name)

    out = cv2.cvtColor(a, cv2.COLOR_BGR2RGB)
    return Image.fromarray(out), damage_names


# ============================================================
# Split helpers
# ============================================================

def build_splits(records: List[dict]):
    random.shuffle(records)

    n = len(records)
    n_train = int(n * SPLIT_RATIOS[0])
    n_val = int(n * SPLIT_RATIOS[1])

    return {
        "train": records[:n_train],
        "validation": records[n_train:n_train + n_val],
        "test": records[n_train + n_val:]
    }


# ============================================================
# Main pipeline
# ============================================================

def main():
    from datasets import load_dataset

    reset_dirs()

    print("=" * 70)
    print("Step 1/5 - Streaming source barcode images from Hugging Face")
    print("=" * 70)
    print(f"Dataset           : {HF_DATASET_NAME}")
    print(f"Split             : {HF_SPLIT}")
    print(f"Target PASS crops : {TARGET_PASS_IMAGES}")
    print(f"Max streamed imgs : {MAX_STREAM_IMAGES}")
    print(f"pyzbar available  : {PYZBAR_AVAILABLE}")
    print()

    ds = load_dataset(
        HF_DATASET_NAME,
        split=HF_SPLIT,
        streaming=True,
        columns=["pixel_values"],
    ).batch(1)

    print("=" * 70)
    print("Step 2/5 - Extracting good barcode crops for PASS")
    print("=" * 70)

    collected_pass = 0
    inspected = 0
    crop_metadata = []

    with tqdm(total=TARGET_PASS_IMAGES, desc="Cropping PASS", unit="crop") as pbar:
        for batch in ds:
            if inspected >= MAX_STREAM_IMAGES:
                break

            inspected += 1

            try:
                pil_img = batch["pixel_values"][0].convert("RGB")
            except Exception:
                continue

            crop = extract_barcode_crop(pil_img)
            if crop is None:
                continue

            save_path = os.path.join(TMP_PASS_DIR, f"pass_{collected_pass:05d}.jpg")
            crop.save(save_path, quality=JPEG_QUALITY)

            crop_metadata.append({
                "pass_file": save_path,
                "source_index": inspected - 1,
                "source_dataset": HF_DATASET_NAME,
                "source_split": HF_SPLIT
            })

            collected_pass += 1
            pbar.update(1)

            if collected_pass >= TARGET_PASS_IMAGES:
                break

    print()
    print(f"Inspected source images : {inspected}")
    print(f"Collected PASS crops    : {collected_pass}")

    if collected_pass == 0:
        print("No usable barcode crops found.")
        print("Try relaxing thresholds or install pyzbar for stronger localization.")
        return

    print()
    print("=" * 70)
    print("Step 3/5 - Generating FAIL crops with synthetic physical damage")
    print("=" * 70)

    pass_paths = sorted(os.path.join(TMP_PASS_DIR, f) for f in os.listdir(TMP_PASS_DIR))
    fail_paths = []

    with tqdm(total=len(pass_paths), desc="Generating FAIL", unit="img") as pbar:
        for idx, src_path in enumerate(pass_paths):
            try:
                src = Image.open(src_path).convert("RGB")
                dmg, damage_names = apply_damage(src)
                dst_path = os.path.join(TMP_FAIL_DIR, f"fail_{idx:05d}.jpg")
                dmg.save(dst_path, quality=JPEG_QUALITY)

                crop_metadata[idx]["fail_file"] = dst_path
                crop_metadata[idx]["damage_types"] = damage_names

                fail_paths.append(dst_path)

                src.close()
                dmg.close()
                pbar.update(1)

            except Exception as e:
                print(f"Warning: failed on {src_path}: {e}")

    print(f"\nGenerated FAIL crops : {len(fail_paths)}")

    print()
    print("=" * 70)
    print("Step 4/5 - Building final PASS/FAIL splits")
    print("=" * 70)

    pass_records = [{"src": p, "label": 1, "label_name": "PASS"} for p in pass_paths]
    fail_records = [{"src": p, "label": 0, "label_name": "FAIL"} for p in fail_paths]

    all_records = pass_records + fail_records
    splits = build_splits(all_records)

    manifest = {
        "config": {
            "seed": SEED,
            "hf_dataset_name": HF_DATASET_NAME,
            "hf_split": HF_SPLIT,
            "target_pass_images": TARGET_PASS_IMAGES,
            "max_stream_images": MAX_STREAM_IMAGES,
            "final_img_size": list(FINAL_IMG_SIZE),
            "split_ratios": {
                "train": SPLIT_RATIOS[0],
                "validation": SPLIT_RATIOS[1],
                "test": SPLIT_RATIOS[2]
            },
            "pyzbar_available": PYZBAR_AVAILABLE,
            "pyzbar_used_for": "localization_only",
            "label_definition": {
                "PASS": "clean barcode crop after quality filtering",
                "FAIL": "same barcode crop with synthetic physical damage"
            }
        },
        "source_crop_metadata": crop_metadata,
        "splits": {}
    }

    print("=" * 70)
    print("Step 5/5 - Saving dataset to disk")
    print("=" * 70)

    for split_name, rows in splits.items():
        split_dir = os.path.join(OUTPUT_DIR, split_name)
        ensure_dir(os.path.join(split_dir, "PASS"))
        ensure_dir(os.path.join(split_dir, "FAIL"))

        n_pass = 0
        n_fail = 0
        manifest["splits"][split_name] = []

        for i, rec in enumerate(tqdm(rows, desc=f"Saving {split_name}", unit="img")):
            dst = os.path.join(split_dir, rec["label_name"], f"{i:05d}.jpg")
            shutil.copy2(rec["src"], dst)

            manifest["splits"][split_name].append({
                "file": dst,
                "label": rec["label"],
                "label_name": rec["label_name"]
            })

            if rec["label"] == 1:
                n_pass += 1
            else:
                n_fail += 1

        print(f"{split_name:12s}: total={len(rows):5d} | PASS={n_pass:4d} FAIL={n_fail:4d}")

    with open(os.path.join(OUTPUT_DIR, "manifest.json"), "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    shutil.rmtree(TMP_PASS_DIR, ignore_errors=True)
    shutil.rmtree(TMP_FAIL_DIR, ignore_errors=True)

    print()
    print(f"Saved dataset to: {os.path.abspath(OUTPUT_DIR)}")
    print()
    print("PyTorch usage:")
    print(r'''
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

T = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

train_ds = datasets.ImageFolder("barcode_dataset_hf_final/train", transform=T)
val_ds   = datasets.ImageFolder("barcode_dataset_hf_final/validation", transform=T)
test_ds  = datasets.ImageFolder("barcode_dataset_hf_final/test", transform=T)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False)

print(train_ds.classes)   # ['FAIL', 'PASS']
print(len(train_ds), len(val_ds), len(test_ds))
''')


if __name__ == "__main__":
    main()

Step 1/5 - Streaming source barcode images from Hugging Face
Dataset           : amaye15/barcodes-google-ocr
Split             : train
Target PASS crops : 3000
Max streamed imgs : 20000
pyzbar available  : True



Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Step 2/5 - Extracting good barcode crops for PASS


Cropping PASS: 100%|█████████████████████████████████████████████████████████████| 3000/3000 [13:30<00:00,  3.70crop/s]


...

train       : total= 4200 | PASS=2111 FAIL=2089


Saving validation: 100%|███████████████████████████████████████████████████████████| 900/900 [00:03<00:00, 280.90img/s]


validation  : total=  900 | PASS= 465 FAIL= 435


Saving test: 100%|█████████████████████████████████████████████████████████████████| 900/900 [00:03<00:00, 263.43img/s]


test        : total=  900 | PASS= 424 FAIL= 476

Saved dataset to: C:\Users\erum8\barcode_dataset_hf_final

PyTorch usage:

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

T = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

train_ds = datasets.ImageFolder("barcode_dataset_hf_final/train", transform=T)
val_ds   = datasets.ImageFolder("barcode_dataset_hf_final/validation", transform=T)
test_ds  = datasets.ImageFolder("barcode_dataset_hf_final/test", transform=T)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False)

print(train_ds.classes)   # ['FAIL', 'PASS']
print(len(train_ds), len(val_ds), len(test_ds))

